# 出席管理・授業後アンケート v2 — Colab 共有用ノート

このノートを上から実行すると、**4画面（生徒用 / 校舎スタッフ用 / 講師・教室担当用 / 校舎QR読取タブレット）を、それぞれ別ウィンドウで開くリンク**を発行します。実データ（CSV）を読み込むので、出席登録・アンケート・各マスタなどの機能を実際に操作して確認できます。

**前提: 最新コードを GitHub に push 済みであること。** Colab はGitHubから取得するため、未pushだと古い画面・古い関数になります。

注意: Colab 上の画面操作はブラウザ内の一時状態です（各自の画面内でのみ反映、「初期データに戻す」で復元可）。

## 共有までに必要な作業（チェックリスト）

### A. リポジトリ準備（PC側・1回）
- [ ] 変更を commit して GitHub に push（`git add -A && git commit -m "v2" && git push`）
- [ ] リポジトリを **Public** にする: GitHub → Settings → 最下部 Danger Zone → Change repository visibility → Public
  - private のままにしたい場合は末尾「private のまま使う場合」セルを使用

### B. Colab で開いて実行
- [ ] https://colab.research.google.com →「ファイル > ノートブックを開く > GitHub」で `YutzMame/tokushin_app_mock` → `mock_app_v2/colab_share_v2.ipynb`
- [ ] 「ランタイム > すべてのセルを実行」→ 発行された4リンクを開く（再実行時は最新を git pull します）

### C. 共有
- [ ] 「ファイル > ドライブにコピーを保存」→ 右上「共有」→「リンクを知っている全員（閲覧者）」→ リンクを送付

## セットアップ（最初に実行）

リポジトリを取得（既存なら git pull で更新）し、表示ヘルパーを読み込みます。

In [ ]:
# === セットアップ: リポジトリ取得/更新 + 表示ヘルパー読込（最初に実行）===
import importlib.util, os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YutzMame/tokushin_app_mock.git"
REPO_DIR = "tokushin_app_mock"
HELPER_REL = Path("mock_app_v2") / "colab_preview_v2.py"

local = HELPER_REL  # リポジトリのルートで開いている場合
if local.exists():
    helper = local
else:
    if Path(REPO_DIR).exists():
        print("既存クローンを最新へ更新します（git pull）...")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print(r.stdout or "", r.stderr or "")
        if r.returncode != 0:
            print("pull に失敗。古いクローンを削除して取り直します...")
            subprocess.run(["rm", "-rf", REPO_DIR])
            print((subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)).stderr or "")
    else:
        print("リポジトリを取得します（git clone）...")
        print((subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)).stderr or "")
    helper = Path(REPO_DIR) / HELPER_REL
    if not helper.exists():
        helper = None

if helper is None:
    raise SystemExit(
        "リポジトリを取得できませんでした。\n"
        "・private の場合: GitHub で Public にする（最も簡単）か、末尾のトークン方式セルを使ってください。\n"
        "・既にローカルにある場合: リポジトリのルートでこのノートを開いてください。"
    )

os.chdir(Path(helper).resolve().parents[1])  # リポジトリのルートへ
spec = importlib.util.spec_from_file_location("colab_preview_v2", "mock_app_v2/colab_preview_v2.py")
preview_v2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preview_v2)
assert hasattr(preview_v2, "serve_links_v2"), "古いコードが読み込まれています。GitHub に最新を push 済みか確認し、再実行してください。"
print("OK: 最新の表示ヘルパーを読み込みました。下のセルを実行してください。")


## 4画面のリンクを発行（CSV込み・別ウィンドウ）

下のセルを実行すると、4画面それぞれの「別ウィンドウで開く」リンクが表示されます。実データCSVを読み込むため、機能を実際に操作できます。

（Colab以外・ローカルで実行した場合は `http://localhost:8000/...` のURLが表示されます。）

In [ ]:
httpd = preview_v2.serve_links_v2()  # 生徒用 / 校舎用 / 講師用 / タブレット の4リンクを発行

## （private のまま使う場合）トークンでクローン

リポジトリを Public にしない場合のみ、下のセルでトークンを使ってクローンします。共有相手にもリポジトリ閲覧権限と各自のトークンが必要になるため、関係者だけで使う想定です。

In [ ]:
# === （private のまま使う場合のみ）Personal Access Token でクローン ===
# GitHub の Settings > Developer settings > Personal access tokens で repo 読取権限のトークンを発行。
# 実行するとトークン入力欄が出ます（ノートには保存されません）。実行後、上のセットアップセルを再実行。
import getpass, subprocess, os
token = getpass.getpass("GitHub Personal Access Token: ")
subprocess.run(["rm", "-rf", "tokushin_app_mock"])
subprocess.run(["git", "clone", f"https://{token}@github.com/YutzMame/tokushin_app_mock.git"])
print("クローン完了したら、上のセットアップセルをもう一度実行してください。")
